# Customer Churn Prediction – Complete Data Analytics & ML Project
**Author:** Akashdeep Singh  
**Dataset:** customer_churn_dataset.csv  
**Target Variable:** Churn (Binary: 0 = Retained, 1 = Churned)

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

print('All imports successful.')

---
## 2. Data Loading & Initial Inspection

In [ ]:
df = pd.read_csv('customer_churn_dataset.csv')

print('=== Dataset Shape ===')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')

print('\n=== First 5 Rows ===')
df.head()

In [ ]:
print('=== Column Data Types ===')
print(df.dtypes)

print('\n=== Statistical Summary ===')
df.describe(include='all')

---
## 3. Data Cleaning

In [ ]:
# ----- 3.1 Missing Values -----
missing = df.isnull().sum()
print('=== Missing Values per Column ===')
print(missing)
print(f'\nTotal missing cells: {missing.sum()}')

In [ ]:
# ----- 3.2 Duplicate Rows -----
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes}')
if dupes > 0:
    df.drop_duplicates(inplace=True)
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicate rows – dataset is clean on this front.')

In [ ]:
# ----- 3.3 Validate Value Ranges / Anomalies -----
checks = {
    'Age': (18, 65),
    'Tenure': (0, 100),
    'Usage Frequency': (0, 30),
    'Support Calls': (0, 20),
    'Payment Delay': (0, 60),
    'Total Spend': (0, 5000),
    'Last Interaction': (1, 30),
}

print('=== Anomaly / Out-of-Range Check ===')
for col, (lo, hi) in checks.items():
    bad = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'  {col}: {bad} values outside [{lo}, {hi}]')

print('\nGender unique:', df['Gender'].unique())
print('Subscription Type unique:', df['Subscription Type'].unique())
print('Contract Length unique:', df['Contract Length'].unique())
print('Churn unique:', df['Churn'].unique())

In [ ]:
# ----- 3.4 Data Type Corrections -----
# Churn is already int (0/1) – good.
# Categorical columns are object – we will encode later for ML.
# Reset index after dedup
df.reset_index(drop=True, inplace=True)

print('Data cleaning complete.')
print(f'Final clean dataset shape: {df.shape}')
df.head(3)

In [ ]:
# ----- 3.5 Churn Class Balance -----
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print('Churn Distribution:')
for k in [0, 1]:
    label = 'Churned' if k == 1 else 'Retained'
    print(f'  {label}: {churn_counts[k]:,}  ({churn_pct[k]:.1f}%)')

---
## 4. Exploratory Data Analysis (EDA) – 5 Charts with Insights

### Chart 1 – Churn Rate by Contract Length
**Question:** Are month-to-month customers more likely to churn?

In [ ]:
churn_contract = (
    df.groupby('Contract Length')['Churn']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
churn_contract.columns = ['Contract Length', 'Churn Rate (%)']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    churn_contract['Contract Length'],
    churn_contract['Churn Rate (%)'],
    color=['#e74c3c', '#f39c12', '#2ecc71'],
    edgecolor='white', width=0.5
)
for bar, val in zip(bars, churn_contract['Churn Rate (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Churn Rate by Contract Length', fontsize=14, fontweight='bold')
ax.set_xlabel('Contract Length')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, churn_contract['Churn Rate (%)'].max() + 10)
sns.despine()
plt.tight_layout()
plt.savefig('chart1_churn_by_contract.png', bbox_inches='tight')
plt.show()

print(churn_contract.to_string(index=False))
print('\n>> INSIGHT: Monthly contract customers have a significantly higher churn rate')
print('   compared to Annual customers. Long-term contracts act as a retention lever.')
print('   Caution: higher churn rate does not mean contracts CAUSE churn — tenure and')
print('   pricing strategy are likely confounding factors.')

### Chart 2 – Support Calls Distribution by Churn Status
**Question:** Do churned customers raise more support calls?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for churn_val, label, color in [(0, 'Retained', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
    subset = df[df['Churn'] == churn_val]['Support Calls']
    ax.hist(subset, bins=range(0, 12), alpha=0.65, label=label,
            color=color, edgecolor='white', density=True)
ax.set_title('Support Calls Distribution – Churned vs Retained', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Support Calls')
ax.set_ylabel('Density')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('chart2_support_calls_churn.png', bbox_inches='tight')
plt.show()

print('Mean Support Calls – Retained:', df[df['Churn']==0]['Support Calls'].mean().round(2))
print('Mean Support Calls – Churned: ', df[df['Churn']==1]['Support Calls'].mean().round(2))
print('\n>> INSIGHT: Churned customers tend to have a higher frequency of support calls.')
print('   This may reflect unresolved issues or dissatisfaction. However, correlation')
print('   does not imply causation — heavy product users also naturally call more.')

### Chart 3 – Churn Rate by Subscription Type
**Question:** Which subscription tier has the highest churn risk?

In [ ]:
churn_sub = (
    df.groupby('Subscription Type')['Churn']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'Churn Rate', 'count': 'Customer Count'})
    .reset_index()
)
churn_sub['Churn Rate (%)'] = (churn_sub['Churn Rate'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: churn rate
axes[0].bar(churn_sub['Subscription Type'], churn_sub['Churn Rate (%)'],
            color=['#3498db', '#9b59b6', '#e67e22'], edgecolor='white', width=0.5)
for i, row in churn_sub.iterrows():
    axes[0].text(i, row['Churn Rate (%)']+0.5, f"{row['Churn Rate (%)']}%",
                 ha='center', fontweight='bold')
axes[0].set_title('Churn Rate by Subscription', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, churn_sub['Churn Rate (%)'].max() + 10)

# Right: volume
axes[1].bar(churn_sub['Subscription Type'], churn_sub['Customer Count'],
            color=['#3498db', '#9b59b6', '#e67e22'], edgecolor='white', width=0.5)
axes[1].set_title('Customer Volume by Subscription', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')

sns.despine()
plt.tight_layout()
plt.savefig('chart3_churn_by_subscription.png', bbox_inches='tight')
plt.show()

print(churn_sub[['Subscription Type', 'Customer Count', 'Churn Rate (%)']].to_string(index=False))
print('\n>> INSIGHT: Basic-tier customers show the highest churn rate. Premium customers')
print('   churn less despite higher price — suggesting perceived value is a retention driver.')
print('   Volume data matters equally; a small % on a large segment is still a big loss.')

### Chart 4 – Tenure vs Total Spend (coloured by Churn)
**Question:** Do high-spend, low-tenure customers churn more?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {0: '#2ecc71', 1: '#e74c3c'}
labels = {0: 'Retained', 1: 'Churned'}

for churn_val in [0, 1]:
    subset = df[df['Churn'] == churn_val]
    ax.scatter(
        subset['Tenure'], subset['Total Spend'],
        alpha=0.25, s=6, color=colors[churn_val], label=labels[churn_val]
    )

ax.set_title('Tenure vs Total Spend by Churn Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Total Spend ($)')
ax.legend(markerscale=4)
sns.despine()
plt.tight_layout()
plt.savefig('chart4_tenure_spend_churn.png', bbox_inches='tight')
plt.show()

print('Mean Tenure  – Retained:', df[df['Churn']==0]['Tenure'].mean().round(1))
print('Mean Tenure  – Churned: ', df[df['Churn']==1]['Tenure'].mean().round(1))
print('Mean Spend   – Retained:', df[df['Churn']==0]['Total Spend'].mean().round(1))
print('Mean Spend   – Churned: ', df[df['Churn']==1]['Total Spend'].mean().round(1))
print('\n>> INSIGHT: Churned customers tend to have lower tenure, meaning early-stage')
print('   customers are at higher risk. Targeted onboarding and early engagement')
print('   programmes could reduce this attrition.')

### Chart 5 – Correlation Heatmap of Numeric Features
**Question:** Which numeric features are most inter-correlated?

In [ ]:
numeric_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
                'Payment Delay', 'Total Spend', 'Last Interaction', 'Churn']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set_title('Correlation Heatmap – Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart5_correlation_heatmap.png', bbox_inches='tight')
plt.show()

churn_corr = corr['Churn'].drop('Churn').sort_values(key=abs, ascending=False)
print('Feature correlations with Churn (sorted by absolute value):')
print(churn_corr.round(3))
print('\n>> INSIGHT: Support Calls, Payment Delay, and Tenure show the strongest')
print('   linear association with Churn. These are the most predictive numeric')
print('   signals. Note: correlation measures linear relationship only and does')
print('   not imply these features cause churn.')

---
## 5. Feature Engineering & ML Preparation

In [ ]:
# ----- 5.1 Drop ID column (irrelevant, no predictive power) -----
ml_df = df.drop(columns=['CustomerID']).copy()
print('Dropped: CustomerID (raw identifier – target leakage risk)')

# ----- 5.2 Encode Categorical Features -----
# Gender: binary encode
ml_df['Gender'] = (ml_df['Gender'] == 'Male').astype(int)

# Subscription Type & Contract Length: ordinal / one-hot
ml_df = pd.get_dummies(ml_df,
                       columns=['Subscription Type', 'Contract Length'],
                       drop_first=False)

print('\nFeatures after encoding:')
print(ml_df.columns.tolist())
print(f'Shape: {ml_df.shape}')

In [ ]:
# ----- 5.3 Define X and y -----
TARGET = 'Churn'
X = ml_df.drop(columns=[TARGET])
y = ml_df[TARGET]

print(f'Features (X): {X.shape[1]} columns, {X.shape[0]:,} rows')
print(f'Target  (y): {y.shape[0]:,} rows  |  Churn rate: {y.mean()*100:.1f}%')

In [ ]:
# ----- 5.4 Train-Test Split (80/20, stratified) -----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training set : {X_train.shape[0]:,} rows')
print(f'Test set     : {X_test.shape[0]:,} rows')
print(f'Train churn rate: {y_train.mean()*100:.1f}%')
print(f'Test  churn rate: {y_test.mean()*100:.1f}%')

In [ ]:
# ----- 5.5 Feature Scaling (for Logistic Regression baseline) -----
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Scaling complete.')

---
## 6. Model Training & Evaluation

### 6.1 Baseline – Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

y_pred_lr   = lr.predict(X_test_sc)
y_proba_lr  = lr.predict_proba(X_test_sc)[:, 1]

print('=== Logistic Regression ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_lr):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred_lr):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_proba_lr):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Retained', 'Churned']))

### 6.2 Primary Model – Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf   = rf.predict(X_test)
y_proba_rf  = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest Classifier ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_proba_rf):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Retained', 'Churned']))

In [ ]:
# ----- Confusion Matrix -----
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, y_pred, model_name in [
    (axes[0], y_pred_lr, 'Logistic Regression'),
    (axes[1], y_pred_rf, 'Random Forest')
]:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Retained', 'Churned'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix\n{model_name}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- ROC Curve Comparison -----
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, y_proba_lr, name='Logistic Regression', ax=ax)
RocCurveDisplay.from_predictions(y_test, y_proba_rf, name='Random Forest',       ax=ax)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Baseline')
ax.set_title('ROC Curve Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
sns.despine()
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

### 6.3 Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
top_features.sort_values().plot(kind='barh', ax=ax, color='#3b82d4', edgecolor='white')
ax.set_title('Top 10 Feature Importances – Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
sns.despine()
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 10 Features:')
print(top_features.round(4))

---
## 7. Model Summary & Business Trade-offs

In [ ]:
metrics = {
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy':  [accuracy_score(y_test, y_pred_lr),  accuracy_score(y_test, y_pred_rf)],
    'Precision': [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_rf)],
    'Recall':    [recall_score(y_test, y_pred_lr),    recall_score(y_test, y_pred_rf)],
    'F1 Score':  [f1_score(y_test, y_pred_lr),        f1_score(y_test, y_pred_rf)],
    'ROC-AUC':   [roc_auc_score(y_test, y_proba_lr),  roc_auc_score(y_test, y_proba_rf)],
}
results_df = pd.DataFrame(metrics).set_index('Model').round(4)
print('=== Model Comparison ===')
print(results_df)

In [ ]:
print("""
=== BUSINESS TRADE-OFFS ===

1. PRECISION vs RECALL:
   - High Recall (minimize false negatives): Catch as many churners as possible.
     Cost = unnecessary retention offers sent to loyal customers (wasteful spend).
   - High Precision (minimize false positives): Only target confirmed churners.
     Cost = miss some real churners who leave without intervention.
   Recommendation: In churn use-cases, RECALL is typically more important —
   the cost of losing a customer > cost of one extra retention offer.

2. THRESHOLD TUNING:
   The default decision threshold (0.50) can be lowered (e.g., 0.35) to
   increase recall and catch more churners at the cost of more false positives.

3. MODEL CHOICE:
   Random Forest outperforms Logistic Regression on ROC-AUC and F1,
   making it the recommended production model. LR remains valuable for
   interpretability and regulatory explainability requirements.
""")

---
## 8. Business Insights & Actionable Recommendations

In [ ]:
print("""
=== ACTIONABLE BUSINESS RECOMMENDATIONS ===

1. CONTRACT UPGRADE CAMPAIGNS:
   Monthly-contract customers churn at a notably higher rate than Annual customers.
   ACTION: Offer a discounted Annual/Quarterly upgrade to new Monthly subscribers
   within the first 60 days (early tenure is highest-risk window).

2. SUPPORT CALL ESCALATION PROTOCOL:
   Customers with 5+ support calls are disproportionately represented in churners.
   ACTION: Trigger an automated 'health-check' outreach after the 4th support call.
   Assign a dedicated success manager for high-call customers in Premium/Standard tier.

3. PAYMENT DELAY EARLY WARNING:
   Payment delay is a strong signal correlated with churn.
   ACTION: When a customer crosses a 15-day payment delay threshold, proactively
   reach out with flexible payment plan offers before they cancel.

4. BASIC-TIER UPGRADE NUDGES:
   Basic subscribers show the highest churn rate.
   ACTION: At 90 days, present a value-comparison of Standard/Premium features.
   Consider a 30-day free trial upgrade to demonstrate value.

5. RESOURCE-CONSTRAINED PRIORITISATION STRATEGY:
   Score all customers monthly using the Random Forest model's predicted probability.
   Segment into 3 action tiers:
     - P(Churn) >= 0.70  → High Risk: Dedicated CSM outreach + personalized offer
     - P(Churn) 0.45-0.69 → Medium Risk: Automated email sequence + upgrade offer
     - P(Churn) < 0.45   → Low Risk: Standard comms, no extra cost
   This ensures the retention budget is concentrated where ROI is highest.
""")

---
## 9. Risk-Scoring on Full Dataset (Deployment Preview)

In [ ]:
# Score entire dataset
all_proba = rf.predict_proba(X)[:, 1]
df_scored = df[['CustomerID', 'Age', 'Tenure', 'Support Calls',
                'Payment Delay', 'Subscription Type', 'Contract Length',
                'Total Spend', 'Churn']].copy()
df_scored['Churn_Probability'] = all_proba
df_scored['Risk_Tier'] = pd.cut(
    all_proba,
    bins=[0, 0.45, 0.70, 1.0],
    labels=['Low', 'Medium', 'High']
)

print('Risk Tier Distribution:')
print(df_scored['Risk_Tier'].value_counts())

print('\nSample High-Risk Customers (P(Churn) >= 0.70):')
high_risk = df_scored[df_scored['Risk_Tier'] == 'High'].sort_values(
    'Churn_Probability', ascending=False
)
print(high_risk[['CustomerID', 'Tenure', 'Support Calls', 'Payment Delay',
                 'Total Spend', 'Contract Length', 'Churn_Probability']].head(10).to_string(index=False))

In [ ]:
# Save scored output
df_scored.to_csv('customer_churn_scored.csv', index=False)
print('Scored dataset saved to: customer_churn_scored.csv')
print('\n=== PROJECT COMPLETE ===')